# 13. Readers: solutions

Exercises of [chapter 13](../../13_readers.md). Run the setup cell first.

These are the solutions; try the exercise notebook first.

In [ ]:
# Setup: run this cell first, in every notebook
import os, subprocess, sys
from pathlib import Path

# find the repository: from this folder upwards, the first one holding data_acquisition/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_acquisition").is_dir())
sys.path.insert(0, str(ROOT))                  # so that Python can import the pipeline
SAMPLE = ROOT / "output/manual/sample_root"    # the course's invented dataset
if not SAMPLE.exists():                        # generated once, with this same Python
    subprocess.run([sys.executable, str(ROOT / "docs/manual/tools/make_sample.py"), "--out", str(SAMPLE)], check=True)
DATASET = SAMPLE / "repo/doi_10.5555_training.pond"    # the sample's dataset folder
EXTRACT = ROOT / "docs/manual/data/montana_extract"    # a real piece of Rumney Ranch
os.environ["BEAVER_SITES"] = str(SAMPLE / "sites.yaml")    # use the sample's site registry, not the real one
import warnings; warnings.simplefilter("ignore")           # hide warnings; remove this line when debugging
print("repository:", ROOT); print("sample data root:", SAMPLE)

In [ ]:
import json, shutil, tempfile, textwrap
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
from data_acquisition.readers import (RECORD_FILES, RECORDS, ckan, entity_bands, files, geotiff, hydroshare, pangaea,
                                      photos, table, vector, zenodo)

READERS = {"zenodo": zenodo, "hydroshare": hydroshare, "ckan": ckan, "pangaea": pangaea}
MD5 = "ab" * 16
RECORD_TEXTS = {        # small records, in the form each repository serves them
    "zenodo": json.dumps({
        "doi": "10.5281/zenodo.2", "conceptdoi": "10.5281/zenodo.1",
        "metadata": {"title": "A survey", "license": {"id": "cc-by-4.0"}, "creators": [{"name": "Rossi, Anna"}],
                     "dates": [{"start": "2019-06-01", "end": "2019-06-30"}], "description": "<p>Flown in June.</p>"},
        "files": [{"key": "ortho.tif", "size": 10, "id": "f1", "checksum": f"md5:{MD5}",
                   "links": {"self": "https://zenodo.org/f/ortho.tif"}}]}),
    "hydroshare": json.dumps({
        "metadata": {"title": "Reach survey", "citation": "Doe (2020). https://doi.org/10.4211/hs.abc",
                     "rights": {"url": "http://creativecommons.org/licenses/by/4.0/", "statement": "CC BY"},
                     "creators": [{"name": "Anna Rossi"}], "abstract": "An abstract",
                     "period_coverage": {"start": "2020-01-01", "end": "2020-12-31"},
                     "spatial_coverage": {"type": "point", "east": -110.5, "north": 44.1}},
        "files": [{"url": "https://x/data/contents/a.csv", "size": 1, "checksum": MD5.upper()},
                  {"url": "https://x/data/contents/b.tif", "size": 2, "checksum": "cd" * 16 + "-2"}]}),
    "ckan": json.dumps({"package": {"result": {
        "id": "pkg-1", "title": "Dams", "license_id": "cc-by", "license_title": "CC BY", "notes": "Some notes",
        "time_period_coverage_start": "0001-01-01", "time_period_coverage_end": "2004-12-07T00:00:00",
        "spatial": json.dumps({"type": "Point", "coordinates": [-85.0, 50.0]}), "organization": {"title": "Ministry"},
        "resources": [{"id": "r1", "format": "ZIP", "url": "https://x/my%20dams.zip", "size": "100", "hash": MD5},
                      {"id": "r2", "format": "HTML", "url": "https://x/viewer"}]}}}),
    "pangaea": ("/* DATA DESCRIPTION:\nCitation:\tRossi, A; Doe, J (2021): Clyst survey [dataset]. PANGAEA, "
                "https://doi.org/10.1594/PANGAEA.1\nLicense:\tCreative Commons Attribution 4.0 (CC-BY-4.0)\n*/\n"
                "Binary\tBinary (Hash)\tBinary (MIME type)\tBinary (Size)\n"
                f"ortho.tif\t{MD5.upper()}\timage/tiff\t1.9 kBytes\nnote.txt\t{MD5}\ttext/plain\t12 Bytes\n"),
}

**1. Rasters.** Read the header of the sample's orthomosaic and of `dem_2022.asc` with `geotiff.info`. Which bands does each declare, and what does the grid lack? Is `README.txt` a raster?

> **From the chapter.** §1: `geotiff.py` reads headers only, and recognises a raster by its extension or by opening it.
>
> **Code.** `geotiff.info`, `geotiff.declared_bands` and `geotiff.is_raster`, in [`data_acquisition/readers/geotiff.py`](../../../../data_acquisition/readers/geotiff.py).

In [ ]:
for name in ("ortho_2023-07-12.tif", "dem_2022.asc"):
    h = geotiff.info(DATASET / name)
    print(name, {k: h[k] for k in ("driver", "epsg", "has_crs", "count", "dtypes", "nodata", "colorinterp")})
    print("   declared bands:", geotiff.declared_bands(h))
print("README.txt is a raster:", geotiff.is_raster(DATASET / "README.txt"))
# The orthomosaic declares red, green, blue, alpha. The grid declares nothing (one grey band) and has no CRS:
# the summary must give it (crs_from_summary, chapter 10).

**2. Vectors.** Read the extract's `dams.gpkg` (26 July) with `vector.info`. Where did its vertical CRS go, and which geometry type is reported?

> **From the chapter.** §1: `vector.info` returns per layer the geometry, features, fields and CRS, with a compound CRS split.
>
> **Code.** `vector.info`, in [`data_acquisition/readers/vector.py`](../../../../data_acquisition/readers/vector.py).

In [ ]:
info = vector.info(EXTRACT / "2018-07-26/annotations/dams.gpkg")
layer = info["layers"][0]
print({k: layer[k] for k in ("name", "geometry_type", "features", "epsg", "vertical_crs")})
print("horizontal CRS:", layer["crs"][:90], "...")
# The compound CRS is split: the horizontal part is "crs", the vertical one (NAVD88, US survey feet) goes to
# vertical_crs. The type is the one GDAL reads, without M (chapter 1).

**3. Records.** Read the sample's `metadata.xml` with the reader `RECORDS` gives for it. What does it say about each file's checksum?

> **From the chapter.** §2: one reader per record format, registered in `RECORDS`, and all return the same shape.
>
> **Code.** `RECORD_FILES` and `RECORDS`, in [`data_acquisition/readers/__init__.py`](../../../../data_acquisition/readers/__init__.py); for this file the reader is in [`data_acquisition/readers/eml.py`](../../../../data_acquisition/readers/eml.py).

In [ ]:
fmt = RECORD_FILES["metadata.xml"]
record = RECORDS[fmt]["read"]((DATASET / "metadata.xml").read_text())
print(fmt, "|", record["title"], "|", record["licence"], "|", record["dates"])
for name, e in record["entities"].items():
    print(f"  {name:28s} {e['size']} bytes, checksum {e['checksum']}")

**4. ArcGIS sidecars.** Read the four `original_*.arcgis.aux.xml` of the extract with `geotiff.arcgis_adjustment`; fit a first-order polynomial through each one's control points and compute the residuals. How do the chunks and the full tiles compare, and what does that say about [chapter 11](../../11_field_data.md)'s alignment results?

> **From the chapter.** §3, and [chapter 3](../../03_drone_surveys.md), §5: ArcGIS keeps its adjustment, with its control points, in a sidecar.
>
> **Code.** `geotiff.arcgis_adjustment`, in [`data_acquisition/readers/geotiff.py`](../../../../data_acquisition/readers/geotiff.py); `numpy.linalg.lstsq` for the fit.

In [ ]:
for aux in sorted(EXTRACT.glob("*/original_*.arcgis.aux.xml")):
    x = geotiff.arcgis_adjustment(aux.read_text())
    design = np.c_[x["source"], np.ones(len(x["source"]))]
    coef = np.linalg.lstsq(design, x["target"], rcond=None)[0]
    residuals = np.hypot(*(design @ coef - x["target"]).T)
    print(f"{aux.parent.name} {aux.name:34s} {x['type']} order {x['order']}, {len(residuals):2d} points, "
          f"rms {np.sqrt((residuals ** 2).mean()):.3f} m, max {residuals.max():.3f} m")
# The chunks fit their few control points to a few centimetres; the full tiles miss theirs by 0.5-0.6 m on average
# (up to 2 m): one affine cannot follow a whole tile's distortion. That is why the annotations, drawn on the chunks,
# sit 0.5-1 m off on the full tiles (chapter 11). The rms is the one in georeference.csv on the drive: an affine
# change of the source coordinates (to pixels, as the field step does) leaves an affine fit's residuals unchanged.

**5. Companions.** Which files does `files.companions` give for `ortho.tif` and for `dams.shp`?

> **From the chapter.** §1, and [chapter 1](../../01_geospatial_data.md), §5: companions are decided from file names only.
>
> **Code.** `files.companions`, in [`data_acquisition/readers/files.py`](../../../../data_acquisition/readers/files.py). The files need not exist.

In [ ]:
print(sorted(files.companions("ortho.tif")))
print(sorted(files.companions("dams.shp")))
print(files.inside_container("Dams.gdb/a00000001.gdbtable"))

**6. Test an edge case.** EXIF stores the altitude as a positive number with a separate flag (`GPSAltitudeRef`: 1 = below sea level). Write a pytest test that makes a photo 400 m below sea level and checks that `photos.exif` reads -400 m. Run it in a scratch folder. What happens?

> **From the chapter.** §4: a reader returns facts about a file. [Chapter 18](../../18_testing.md), §1 explains known bugs and `xfail`.
>
> **Code.** `photos.exif` in [`data_acquisition/readers/photos.py`](../../../../data_acquisition/readers/photos.py), inside a test you write and run with `pytest`; Pillow makes the photo.

In [ ]:
scratch = Path(tempfile.mkdtemp())
test = textwrap.dedent('''
    import pytest
    from PIL import ExifTags, Image
    from data_acquisition.readers import photos

    @pytest.mark.xfail(strict=True, reason="GPSAltitudeRef is not read (course finding 19)")
    def test_altitude_below_sea_level(tmp_path):
        exif = Image.Exif()
        exif[ExifTags.IFD.GPSInfo] = {
            ExifTags.GPS.GPSLatitudeRef: "N", ExifTags.GPS.GPSLatitude: (31.0, 30.0, 0.0),
            ExifTags.GPS.GPSLongitudeRef: "E", ExifTags.GPS.GPSLongitude: (35.0, 30.0, 0.0),
            ExifTags.GPS.GPSAltitudeRef: b"\\x01", ExifTags.GPS.GPSAltitude: 400.0}
        path = tmp_path / "below.jpg"
        Image.new("RGB", (8, 8)).save(path, exif=exif)
        assert photos.exif(path)["alt"] == -400.0
''')
(scratch / "test_edge.py").write_text(test)
# -c: the repository's pytest.ini (it switches off plugins that do not work here), also for a test outside the repo
command = [sys.executable, "-m", "pytest", "-q", "--color=no", "-c", str(ROOT / "pytest.ini"), "--rootdir", str(ROOT)]
out = subprocess.run(command + [str(scratch / "test_edge.py")], cwd=ROOT, capture_output=True, text=True)
print((out.stdout + out.stderr).strip().splitlines()[-1])
# The reader returns +400: the assertion fails, so the test is marked xfail (strict: once the reader is fixed it
# will fail loudly, a reminder to remove the mark). A test that documents a bug, with the bug logged, is how an
# edge case enters the suite before its fix (chapter 18).

**7. One shape from five standards.** `RECORD_TEXTS`, in the notebook, holds small records in the Zenodo, HydroShare, CKAN and PANGAEA formats. Read each with its reader: which keys do all share, and which entity did CKAN skip? Which size is only approximate? Then try `ckan.file_name`, `hydroshare.resource_id` and `readers.entity_bands`.

> **From the chapter.** §2: every record reader returns the same shape.
>
> **Code.** `READERS` and `RECORD_TEXTS`, defined above. The readers are in [`data_acquisition/readers/zenodo.py`](../../../../data_acquisition/readers/zenodo.py), [`data_acquisition/readers/hydroshare.py`](../../../../data_acquisition/readers/hydroshare.py), [`data_acquisition/readers/ckan.py`](../../../../data_acquisition/readers/ckan.py) and [`data_acquisition/readers/pangaea.py`](../../../../data_acquisition/readers/pangaea.py); `entity_bands` is in [`data_acquisition/readers/__init__.py`](../../../../data_acquisition/readers/__init__.py).

In [ ]:
records = {name: READERS[name].read(text) for name, text in RECORD_TEXTS.items()}
for name, r in records.items():
    print(name, {k: r[k] for k in ("package_id", "title", "licence", "creators", "dates", "bounds")})
    print("   ", {f: (e["size"], e["checksum"], e.get("size_exact")) for f, e in r["entities"].items()})
print("shared keys:", sorted(set.intersection(*(set(r) for r in records.values()))))
# CKAN skipped the "viewer" link (format HTML: a web page, not a file) and unquoted my%20dams.zip. PANGAEA rounds
# sizes ("1.9 kBytes"), so size_exact is False and the builder does not compare them. HydroShare gives an MD5 for one
# file and a multipart ETag for the other: same record, two checksum forms.
print(ckan.file_name("https://x/a%20b.zip?download=1"))
print(hydroshare.resource_id("https://doi.org/10.4211/hs." + "f" * 32))
print(entity_bands({"attributes": [{"name": "Red", "codes": []}, {"name": "NIR", "codes": []}]}))
print(entity_bands({"attributes": [{"name": "Band type", "codes": [("1", "Green"), ("2", "Blue")]}]}))
print(entity_bands({"attributes": [{"name": "elevation", "codes": []}]}))
# entity_bands: attribute names, else the codes of the attribute; "elevation" names the values, not a band.

**8. Tables.** Write a CSV with a title line, coordinates and a row without coordinates, and one without coordinates at all. What do `table.read`, `table.info` and `table.rows_info` say? What happens if the spec names a column that does not exist?

> **From the chapter.** §1: `table.py` reads CSV and TSV as the summary's `table` spec says.
>
> **Code.** `table.read`, `table.info` and `table.rows_info`, in [`data_acquisition/readers/table.py`](../../../../data_acquisition/readers/table.py).

In [ ]:
scratch = Path(tempfile.mkdtemp())
(scratch / "gcps.csv").write_text("Surveyed points\nGCP,Northing,Easting\n"
                                  "1,5420030,368010\n2,5420031,368011\n3,,\n")
(scratch / "reaches.csv").write_text("ReachID,value\n1,0.5\n2,1.2\n")
spec = {"skip": 1, "x": "Easting", "y": "Northing"}
print("read:", table.read(scratch / "gcps.csv", spec).shape, "rows, columns")
layer = table.info(scratch / "gcps.csv", spec, "EPSG:32612")["layers"][0]
print({k: layer[k] for k in ("geometry_type", "features", "epsg", "bounds", "rows_without_coordinates")})
rows = table.rows_info(scratch / "reaches.csv", {"skip": 0})
print("no coordinates:", rows["features"], "rows,", rows["layers"][0]["fields"], "| spatial:", rows["spatial"])
print("info gives the same:", table.info(scratch / "reaches.csv", {"skip": 0}, "EPSG:32612")["spatial"])
try:
    table.read(scratch / "gcps.csv", {"skip": 1, "x": "East", "y": "North"})
except KeyError as e:
    print("KeyError:", str(e)[:70])
# The same shape as vector.info, so a table becomes a vector item. The CRS is an argument: tables rarely state one.

**9. The small helpers.** `geotiff.sample` on the DSM, with two sizes. `geotiff.unapplied_georeference` on a copy of the extract's chunk, before and after putting its original ArcGIS sidecar next to it; is that sidecar a companion (`files.is_companion`)? `vector.is_vector` and `photos.is_photo_folder` on a few paths; `photos.info` on the sample's photos; `vector.one_layer` on a two-layer GeoPackage.

> **From the chapter.** §1: the helpers of each reader, and §3 for `unapplied_georeference`.
>
> **Code.** `geotiff.sample` and `geotiff.unapplied_georeference` in [`data_acquisition/readers/geotiff.py`](../../../../data_acquisition/readers/geotiff.py), `files.is_companion` in [`data_acquisition/readers/files.py`](../../../../data_acquisition/readers/files.py), `vector.is_vector` and `vector.one_layer` in [`data_acquisition/readers/vector.py`](../../../../data_acquisition/readers/vector.py), `photos.is_photo_folder` and `photos.info` in [`data_acquisition/readers/photos.py`](../../../../data_acquisition/readers/photos.py).

In [ ]:
scratch = Path(tempfile.mkdtemp())
sampled = geotiff.sample(DATASET / "dsm_2023-07-12.tif")
print(type(sampled).__name__, sampled.shape, "| masked:", int(sampled.mask.sum()), "| 10:",
      geotiff.sample(DATASET / "dsm_2023-07-12.tif", size=10).shape)
# The central window of every band, nodata masked: never more than 1024 x 1024, whatever the file's size.

chunk = scratch / "chunk.tif"
shutil.copyfile(EXTRACT / "2018-07-26/orthomosaic_chunk.tif", chunk)
print("no sidecar:", geotiff.unapplied_georeference(chunk))
sidecar = scratch / "chunk.tif.aux.xml"
shutil.copyfile(EXTRACT / "2018-07-26/original_chunk.arcgis.aux.xml", sidecar)
print("with it:   ", geotiff.unapplied_georeference(chunk))
print("companion:", files.is_companion(sidecar), "| the raster itself:", files.is_companion(chunk))

print("vector:", vector.is_vector(DATASET / "dams_2023-07-12.gpkg"), vector.is_vector(DATASET / "metadata.xml"))
print("photos:", photos.is_photo_folder(DATASET / "photos_2023-07-12"), photos.is_photo_folder(scratch))
print({k: v for k, v in photos.info(DATASET / "photos_2023-07-12").items() if k != "positions"})

two = scratch / "two.gpkg"
for name, n in (("a", 2), ("b", 3)):
    gpd.GeoDataFrame({"id": range(n)}, geometry=[Point(368010 + i, 5420030) for i in range(n)],
                     crs="EPSG:32612").to_file(two, layer=name)
info = vector.info(two)
one = vector.one_layer(info, "b")
print(info["features"], "features in all;", one["features"], "in", [layer["name"] for layer in one["layers"]])